# LeetCode 1304 — Find N Unique Integers Sum up to Zero bằng Python

**Link đề:** https://leetcode.com/problems/find-n-unique-integers-sum-up-to-zero/
**Độ khó:** Easy · **Chủ đề:** Array, Math (xây dựng mảng — *constructive*)

```python
class Solution:
    def sumZero(self, n: int) -> list[int]:
        ...
```

## Mục tiêu học
1. Hiểu dạng bài **"trả về một đáp án bất kỳ hợp lệ"** (constructive): không phải *tìm* mà là *tự xây* đáp án.
2. Ôn các công cụ Python: `list`, `append`, `range` (có bước nhảy, số âm), `sum`, `set`, `len`, list comprehension.
3. Nắm 4 cách xây mảng: cặp đối xứng `±x`, `range` đối xứng, "bù số cuối", và công thức `2*i - n + 1`.
4. Biết **tự viết hàm kiểm tra** (validator) cho bài có nhiều đáp án đúng — vì không thể so sánh `==` với một output cố định.
5. Trace, chứng minh tính đúng, phân tích Time/Space, luyện biến thể.

**Cách học:** đọc markdown → đoán output cell kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài — xây `n` số khác nhau có tổng bằng 0

Cho số nguyên `n`. Trả về **một mảng bất kỳ** gồm `n` số nguyên **đôi một khác nhau** (unique) sao cho tổng của chúng bằng `0`.

| Input `n` | Một output hợp lệ | Các output khác cũng đúng |
|---:|---|---|
| `5` | `[-7,-1,1,3,4]` | `[-5,-1,1,2,3]`, `[-3,-1,2,-2,4]` |
| `3` | `[-1,0,1]` | `[-2,0,2]`, `[1,2,-3]` |
| `1` | `[0]` | (chỉ có `[0]`) |

**Constraints:** `1 <= n <= 1000`.

Ba điều kiện của một đáp án đúng:
1. Có **đúng `n`** phần tử.
2. Các phần tử **khác nhau từng đôi**.
3. **Tổng bằng 0**.

Câu hỏi trước khi code:
- Vì sao `n = 1` chỉ có một đáp án duy nhất?
- Nếu `n` chẵn, có được dùng số `0` không? Có *bắt buộc* dùng không?
- Làm sao để chấm điểm đáp án của chính mình khi có rất nhiều đáp án đúng?

## 2. Nền tảng Python cần dùng

### 2.1. `list` và `append`

`list` là dãy có thứ tự, viết trong `[...]`. `append(x)` thêm `x` vào **cuối** list, list tự dài thêm 1.

In [1]:
result = []
print("ban đầu:", result, "len =", len(result))
result.append(3)
result.append(-3)
print("sau 2 lần append:", result, "len =", len(result))

ban đầu: [] len = 0
sau 2 lần append: [3, -3] len = 2


### 2.2. `sum(list)` — cộng tất cả phần tử

`sum([1, -1, 2])` trả về `2`. Với list rỗng, `sum([])` trả về `0`.

In [2]:
print(sum([1, -1, 2]))
print(sum([-7, -1, 1, 3, 4]))
print(sum([]))

2
0
0


### 2.3. `set` — kiểm tra "khác nhau"

`set(...)` bỏ hết phần tử trùng. Nên **`len(set(arr)) == len(arr)`** nghĩa là "mọi phần tử trong `arr` đều khác nhau".

Ví dụ `[1, 2, 2]` → `set` là `{1, 2}` có 2 phần tử, ít hơn 3 → có trùng.

In [3]:
a = [1, 2, 2]
b_ = [1, 2, 3]
print(set(a), len(set(a)), "vs", len(a), "-> unique?", len(set(a)) == len(a))
print(set(b_), len(set(b_)), "vs", len(b_), "-> unique?", len(set(b_)) == len(b_))

{1, 2} 2 vs 3 -> unique? False
{1, 2, 3} 3 vs 3 -> unique? True


### 2.4. `range(start, stop, step)` — dãy số, kể cả bước nhảy và số âm

- `range(1, 4)` → `1, 2, 3` (không lấy `stop`).
- `range(-2, 3)` → `-2, -1, 0, 1, 2`.
- `range(1-n, n, 2)` → nhảy 2 đơn vị một lần (sẽ dùng ở cách 4).

`range` chưa phải list; muốn xem thì bọc `list(...)`.

In [4]:
print(list(range(1, 4)))
print(list(range(-2, 3)))
n = 4
print("range(1-n, n, 2) với n=4:", list(range(1 - n, n, 2)))
n = 5
print("range(1-n, n, 2) với n=5:", list(range(1 - n, n, 2)))

[1, 2, 3]
[-2, -1, 0, 1, 2]
range(1-n, n, 2) với n=4: [-3, -1, 1, 3]
range(1-n, n, 2) với n=5: [-4, -2, 0, 2, 4]


### 2.5. `//` (chia lấy nguyên) và `%` (chia lấy dư)

- `n // 2`: số cặp có thể tạo từ `n` phần tử. `5 // 2 = 2`, `4 // 2 = 2`.
- `n % 2`: bằng `1` nếu `n` lẻ, `0` nếu chẵn.

Hai phép này giúp tách bài thành "các cặp" + "có dư một phần tử hay không".

In [5]:
for n in [1, 2, 3, 4, 5]:
    print(f"n={n}: số cặp n//2 = {n // 2}, lẻ? n%2 = {n % 2}")

n=1: số cặp n//2 = 0, lẻ? n%2 = 1
n=2: số cặp n//2 = 1, lẻ? n%2 = 0
n=3: số cặp n//2 = 1, lẻ? n%2 = 1
n=4: số cặp n//2 = 2, lẻ? n%2 = 0
n=5: số cặp n//2 = 2, lẻ? n%2 = 1


### 2.6. List comprehension — viết vòng lặp tạo list trong 1 dòng

```python
[biểu_thức for i in range(...)]
```

tương đương vòng `for` + `append`. Ví dụ `[i * i for i in range(4)]` → `[0, 1, 4, 9]`.

In [6]:
squares = []
for i in range(4):
    squares.append(i * i)
print("vòng for:", squares)
print("comprehension:", [i * i for i in range(4)])

vòng for: [0, 1, 4, 9]
comprehension: [0, 1, 4, 9]


## 3. Hàm kiểm tra đáp án (validator) — viết TRƯỚC khi giải

Đề nói *"return any array"*: có rất nhiều đáp án đúng. Vì vậy **không được** test kiểu `assert sumZero(5) == [-7,-1,1,3,4]` — đáp án của ta có thể khác nhưng vẫn đúng.

Thay vào đó, ta viết hàm `is_valid(arr, n)` kiểm tra đủ 3 điều kiện của đề. Đây là thói quen quan trọng cho mọi bài *constructive*.

In [7]:
def is_valid(arr: list[int], n: int) -> bool:
    return (
        len(arr) == n               # đúng n phần tử
        and len(set(arr)) == n      # đôi một khác nhau
        and sum(arr) == 0           # tổng bằng 0
    )

print(is_valid([-7, -1, 1, 3, 4], 5))   # True (ví dụ đề)
print(is_valid([-1, 0, 1], 3))          # True
print(is_valid([0, 0], 2))              # False: trùng
print(is_valid([1, -1], 3))             # False: thiếu phần tử
print(is_valid([1, 2], 2))              # False: tổng khác 0

True
True
False
False
False


## 4. Cách trực giác / baseline — thử và sửa

Người mới hay nghĩ: "Lấy các số `1, 2, ..., n-1`, rồi số cuối là **âm tổng** của chúng để bù về 0".

Ví dụ `n = 4`: lấy `1, 2, 3`, tổng là `6`, số cuối là `-6` → `[1, 2, 3, -6]`. Tổng = 0, các số khác nhau.

Đây là **cách "bù số cuối"**. Có ổn không? Hãy tự kiểm tra với vài `n`, đặc biệt `n = 1`.

In [8]:
def sum_zero_fill_last(n: int) -> list[int]:
    arr = list(range(1, n))   # 1, 2, ..., n-1
    arr.append(-sum(arr))     # số cuối bù cho tổng = 0
    return arr

for n in [1, 2, 3, 4, 5]:
    arr = sum_zero_fill_last(n)
    print(n, arr, "hợp lệ?", is_valid(arr, n))

1 [0] hợp lệ? True
2 [1, -1] hợp lệ? True
3 [1, 2, -3] hợp lệ? True
4 [1, 2, 3, -6] hợp lệ? True
5 [1, 2, 3, 4, -10] hợp lệ? True


### 4.1. Vì sao "bù số cuối" luôn khác nhau?

- Với `n = 1`: `range(1, 1)` rỗng, tổng `0`, số cuối là `-0 = 0` → `[0]`. Đúng.
- Với `n >= 2`: các số `1..n-1` đều **dương**, nên tổng của chúng dương → số cuối **âm**, không thể trùng số dương nào. Đúng.

Nhược điểm: số cuối lớn dần `≈ n²/2`. Với `n = 1000` số cuối là `-499500`. Python không sợ tràn số, nhưng ở ngôn ngữ khác (int 32-bit) với `n` lớn hơn nhiều có thể tràn. Ngoài ra ta phải **tính tổng**, tức là thêm một lượt duyệt.

In [9]:
arr = sum_zero_fill_last(1000)
print("phần tử cuối với n=1000:", arr[-1])
print("hợp lệ?", is_valid(arr, 1000))

phần tử cuối với n=1000: -499500
hợp lệ? True


### 4.2. Một baseline "ngây thơ" sai — để thấy vì sao phải cẩn thận

Có người thử: "lấy `n` số `0`". Tổng bằng 0 nhưng **trùng nhau** → sai khi `n >= 2`.

Người khác thử: "lấy `-1, 1` lặp lại". Ví dụ `n = 4` → `[-1, 1, -1, 1]` — vẫn trùng. Bài học: điều kiện **unique** mới là điều kiện khó, tổng 0 thì dễ.

In [10]:
def wrong_all_zero(n):
    return [0] * n

def wrong_repeat_pair(n):
    return [-1, 1] * (n // 2) + [0] * (n % 2)

for n in [1, 2, 4]:
    print(n, wrong_all_zero(n), is_valid(wrong_all_zero(n), n), "|",
          wrong_repeat_pair(n), is_valid(wrong_repeat_pair(n), n))

1 [0] True | [0] True
2 [0, 0] False | [-1, 1] True
4 [0, 0, 0, 0] False | [-1, 1, -1, 1] False


## 5. Nút thắt và insight — **cặp đối xứng `x` và `-x`**

Nút thắt: muốn tổng bằng 0 mà không cần tính tổng, và các số tự động khác nhau.

**Insight (theo hint của đề):** `x + (-x) = 0`. Nếu ta lấy các cặp `(1, -1), (2, -2), ...` thì:
- Mỗi cặp có tổng 0 ⇒ tổng toàn bộ bằng 0.
- Các số `1, 2, 3, ...` khác nhau và các số `-1, -2, ...` khác nhau, số dương không bao giờ bằng số âm ⇒ unique.

Nếu `n` **lẻ**, sau khi tạo `n // 2` cặp vẫn còn thiếu 1 phần tử → thêm số `0` (tổng không đổi, `0` không trùng với số nào khác 0).

In [11]:
n = 5
pairs = n // 2
print("n =", n, "-> số cặp:", pairs, "| cần thêm 0?", n % 2 == 1)
for x in range(1, pairs + 1):
    print(f"cặp ({x}, {-x}) có tổng {x + (-x)}")

n = 5 -> số cặp: 2 | cần thêm 0? True
cặp (1, -1) có tổng 0
cặp (2, -2) có tổng 0


## 6. Lời giải chính — cặp đối xứng, giảng từng đoạn rồi ghép

**Đoạn 1:** tạo list rỗng `result`.

**Đoạn 2:** duyệt `x` từ `1` đến `n // 2`, mỗi lần `append(x)` và `append(-x)`.

**Đoạn 3:** nếu `n` lẻ, `append(0)`.

**Đoạn 4:** trả về `result`.

In [12]:
# Đoạn 1 + 2 với n = 4
n = 4
result = []
for x in range(1, n // 2 + 1):
    result.append(x)
    result.append(-x)
print("sau các cặp:", result)
# Đoạn 3: n chẵn nên không thêm 0
if n % 2 == 1:
    result.append(0)
print("kết quả:", result, "hợp lệ?", is_valid(result, n))

sau các cặp: [1, -1, 2, -2]
kết quả: [1, -1, 2, -2] hợp lệ? True


In [13]:
def sum_zero_pairs(n: int) -> list[int]:
    result = []
    for x in range(1, n // 2 + 1):
        result.append(x)
        result.append(-x)
    if n % 2 == 1:
        result.append(0)
    return result

for n in [1, 2, 3, 4, 5, 6]:
    arr = sum_zero_pairs(n)
    print(n, arr, is_valid(arr, n))

1 [0] True
2 [1, -1] True
3 [1, -1, 0] True
4 [1, -1, 2, -2] True
5 [1, -1, 2, -2, 0] True
6 [1, -1, 2, -2, 3, -3] True


### 6.1. Vì sao là `range(1, n // 2 + 1)` mà không phải `range(n // 2)`?

`range(n // 2)` bắt đầu từ `0` → cặp đầu là `(0, -0)` = `(0, 0)` — **trùng nhau**! Phải bắt đầu từ `1`.

Và vì `range` không lấy `stop`, muốn đi đến `n // 2` thì `stop = n // 2 + 1`.

In [14]:
n = 4
print("range(n//2)        :", list(range(n // 2)), "-> có số 0, cặp (0,0) bị trùng")
print("range(1, n//2 + 1) :", list(range(1, n // 2 + 1)))

range(n//2)        : [0, 1] -> có số 0, cặp (0,0) bị trùng
range(1, n//2 + 1) : [1, 2]


## 7. TRACE từng bước

Bảng trace cho `n = 5` (`n // 2 = 2`, `n` lẻ):

| Bước | x | Hành động | result sau bước | tổng hiện tại |
|---|---|---|---|---|
| 1 | 1 | thêm `1, -1` | `[1, -1]` | 0 |
| 2 | 2 | thêm `2, -2` | `[1, -1, 2, -2]` | 0 |
| 3 | — | `n` lẻ → thêm `0` | `[1, -1, 2, -2, 0]` | 0 |

Điều đáng chú ý: **tổng luôn bằng 0 sau mỗi bước** — đó chính là invariant ta sẽ dùng để chứng minh.

In [15]:
def trace_sum_zero(n: int) -> list[int]:
    print(f"n = {n}, số cặp = {n // 2}, lẻ = {n % 2 == 1}")
    result = []
    for x in range(1, n // 2 + 1):
        result.append(x)
        result.append(-x)
        print(f"  x={x}: thêm ({x}, {-x}) -> result={result}, tổng={sum(result)}")
    if n % 2 == 1:
        result.append(0)
        print(f"  n lẻ: thêm 0 -> result={result}, tổng={sum(result)}")
    print("  trả về:", result)
    return result

trace_sum_zero(5)
print()
trace_sum_zero(4)
print()
trace_sum_zero(1)

n = 5, số cặp = 2, lẻ = True
  x=1: thêm (1, -1) -> result=[1, -1], tổng=0
  x=2: thêm (2, -2) -> result=[1, -1, 2, -2], tổng=0
  n lẻ: thêm 0 -> result=[1, -1, 2, -2, 0], tổng=0
  trả về: [1, -1, 2, -2, 0]

n = 4, số cặp = 2, lẻ = False
  x=1: thêm (1, -1) -> result=[1, -1], tổng=0
  x=2: thêm (2, -2) -> result=[1, -1, 2, -2], tổng=0
  trả về: [1, -1, 2, -2]

n = 1, số cặp = 0, lẻ = True
  n lẻ: thêm 0 -> result=[0], tổng=0
  trả về: [0]


[0]

## 8. Code nộp LeetCode

Bản gọn, không `print`. Chú ý tên method đúng là `sumZero`.

In [16]:
class Solution:
    def sumZero(self, n: int) -> list[int]:
        result = []
        for x in range(1, n // 2 + 1):
            result.append(x)
            result.append(-x)
        if n % 2 == 1:
            result.append(0)
        return result

sol = Solution()
for n in [5, 3, 1]:
    out = sol.sumZero(n)
    print(n, "->", out, "hợp lệ?", is_valid(out, n))

5 -> [1, -1, 2, -2, 0] hợp lệ? True
3 -> [1, -1, 0] hợp lệ? True
1 -> [0] hợp lệ? True


## 9. Các lời giải khác — đi từ cơ bản lên sáng tạo

Bài này có nhiều lời giải đẹp. Ngoài cách 1 (bù số cuối) và cách 2 (cặp đối xứng), còn:

### Cách 3 — `range` đối xứng (một dòng)

- `n` lẻ: `range(-(n//2), n//2 + 1)` cho `-k, ..., -1, 0, 1, ..., k` với `k = n//2`, đủ `2k + 1 = n` số.
- `n` chẵn: bỏ số `0`: `range(-k, 0)` nối `range(1, k + 1)`.

In [17]:
def sum_zero_range(n: int) -> list[int]:
    k = n // 2
    if n % 2 == 1:
        return list(range(-k, k + 1))
    return list(range(-k, 0)) + list(range(1, k + 1))

for n in [1, 2, 3, 4, 5]:
    arr = sum_zero_range(n)
    print(n, arr, is_valid(arr, n))

1 [0] True
2 [-1, 1] True
3 [-1, 0, 1] True
4 [-2, -1, 1, 2] True
5 [-2, -1, 0, 1, 2] True


Ở đây xuất hiện phép **nối list** `+`: `[1, 2] + [3]` → `[1, 2, 3]` (tạo list mới).

In [18]:
print([1, 2] + [3])
print(list(range(-2, 0)) + list(range(1, 3)))

[1, 2, 3]
[-2, -1, 1, 2]


### Cách 4 — công thức `a[i] = 2*i - n + 1` (cấp số cộng đối xứng)

Xét dãy `1 - n, 3 - n, 5 - n, ..., n - 1`: đó là **cấp số cộng** bước 2, đối xứng quanh 0.

- `n = 4`: `-3, -1, 1, 3` → tổng 0.
- `n = 5`: `-4, -2, 0, 2, 4` → tổng 0.

Vì sao tổng bằng 0? Phần tử thứ `i` và phần tử thứ `n-1-i` có tổng `(2i - n + 1) + (2(n-1-i) - n + 1) = 0`. Ghép cặp từ hai đầu vào giữa ⇒ tổng 0. Các phần tử tăng dần (bước 2) ⇒ khác nhau.

Đây cũng chính là `range(1 - n, n, 2)`.

In [19]:
def sum_zero_formula(n: int) -> list[int]:
    return [2 * i - n + 1 for i in range(n)]

for n in [1, 2, 3, 4, 5]:
    arr = sum_zero_formula(n)
    print(n, arr, "== range(1-n, n, 2)?", arr == list(range(1 - n, n, 2)), is_valid(arr, n))

1 [0] == range(1-n, n, 2)? True True
2 [-1, 1] == range(1-n, n, 2)? True True
3 [-2, 0, 2] == range(1-n, n, 2)? True True
4 [-3, -1, 1, 3] == range(1-n, n, 2)? True True
5 [-4, -2, 0, 2, 4] == range(1-n, n, 2)? True True


### Code nộp LeetCode cho các cách còn lại

In [20]:
class SolutionFillLast:
    def sumZero(self, n: int) -> list[int]:
        arr = list(range(1, n))
        arr.append(-sum(arr))
        return arr

class SolutionRange:
    def sumZero(self, n: int) -> list[int]:
        k = n // 2
        if n % 2:
            return list(range(-k, k + 1))
        return list(range(-k, 0)) + list(range(1, k + 1))

class SolutionFormula:
    def sumZero(self, n: int) -> list[int]:
        return list(range(1 - n, n, 2))

for cls in [SolutionFillLast, SolutionRange, SolutionFormula]:
    ok = all(is_valid(cls().sumZero(n), n) for n in range(1, 1001))
    print(cls.__name__, "đúng với mọi n từ 1..1000?", ok)

SolutionFillLast đúng với mọi n từ 1..1000? True
SolutionRange đúng với mọi n từ 1..1000? True
SolutionFormula đúng với mọi n từ 1..1000? True


### Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space (output) | Ghi chú |
|---|---|---|---|---|
| 1. Bù số cuối | `1..n-1` rồi thêm `-tổng` | `O(n)` | `O(n)` | Dễ nghĩ; số cuối lớn ~`n²/2` |
| 2. Cặp `±x` | thêm `x, -x`, lẻ thêm `0` | `O(n)` | `O(n)` | Theo hint, dễ chứng minh nhất |
| 3. `range` đối xứng | `-k..k` (bỏ 0 nếu chẵn) | `O(n)` | `O(n)` | Ngắn, cần nhớ biên |
| 4. Công thức `2i-n+1` | cấp số cộng đối xứng | `O(n)` | `O(n)` | Một dòng, không cần if |

Mọi cách đều `O(n)` vì riêng việc tạo output đã cần `n` bước — không thể nhanh hơn.

## 10. Vì sao lời giải chính đúng? (invariant)

**Invariant:** sau mỗi vòng lặp với giá trị `x`, `result` chứa đúng các số `±1, ±2, ..., ±x`, và `sum(result) == 0`.

- Ban đầu `result = []`, tổng 0 — đúng.
- Mỗi vòng thêm `x` và `-x`: tổng tăng `x - x = 0` → vẫn 0. Hai số mới có trị tuyệt đối `x` lớn hơn mọi số đã có → không trùng.
- Sau vòng lặp có `2 * (n // 2)` phần tử. Nếu `n` chẵn thì đó là `n`. Nếu `n` lẻ thì thiếu 1, ta thêm `0`: tổng không đổi, và `0` khác mọi `±x` với `x >= 1`.

Vậy kết quả có đúng `n` phần tử, unique, tổng 0. ∎

### 10.1. Độ phức tạp

Gọi `n` là input.

- **Time:** `O(n)` — vòng lặp chạy `n // 2` lần, mỗi lần 2 `append` (`O(1)` khấu hao).
- **Space:** `O(n)` cho mảng output (bắt buộc phải trả về). Bộ nhớ phụ ngoài output: `O(1)`.

Không thể tốt hơn `O(n)` vì output có `n` phần tử.

## 11. Bộ test

Vì có nhiều đáp án đúng, mọi test đều dùng `is_valid`. Ta test: ví dụ đề, biên `n = 1`, `n = 2`, `n` lớn nhất `1000`, và **đối chiếu mọi cách** trên toàn miền `1..1000`.

In [21]:
def check(name, arr, n):
    assert is_valid(arr, n), (name, n, arr)
    print(f"{name:<18} n={n:<4} PASS")

sol = Solution()
check("ví dụ 1", sol.sumZero(5), 5)
check("ví dụ 2", sol.sumZero(3), 3)
check("ví dụ 3 (biên)", sol.sumZero(1), 1)
check("chẵn nhỏ nhất", sol.sumZero(2), 2)
check("lớn nhất", sol.sumZero(1000), 1000)
check("lẻ lớn", sol.sumZero(999), 999)

ví dụ 1            n=5    PASS
ví dụ 2            n=3    PASS
ví dụ 3 (biên)     n=1    PASS
chẵn nhỏ nhất      n=2    PASS
lớn nhất           n=1000 PASS
lẻ lớn             n=999  PASS


In [22]:
# Đối chiếu: mọi cách đều phải hợp lệ trên toàn bộ miền constraints
methods = [sum_zero_fill_last, sum_zero_pairs, sum_zero_range, sum_zero_formula]
for f in methods:
    for n in range(1, 1001):
        assert is_valid(f(n), n), (f.__name__, n)
    print(f.__name__, "PASS cho n = 1..1000")

sum_zero_fill_last PASS cho n = 1..1000
sum_zero_pairs PASS cho n = 1..1000
sum_zero_range PASS cho n = 1..1000


sum_zero_formula PASS cho n = 1..1000


In [23]:
# Phản ví dụ: validator phải bắt được các đáp án sai
bad_cases = [
    ([0, 0], 2, "trùng số 0"),
    ([1, -1, 1, -1], 4, "trùng cặp"),
    ([1, 2, 3], 3, "tổng khác 0"),
    ([-1, 1], 3, "thiếu phần tử"),
]
for arr, n, why in bad_cases:
    assert not is_valid(arr, n)
    print("validator bắt được lỗi:", why, arr)

validator bắt được lỗi: trùng số 0 [0, 0]
validator bắt được lỗi: trùng cặp [1, -1, 1, -1]
validator bắt được lỗi: tổng khác 0 [1, 2, 3]
validator bắt được lỗi: thiếu phần tử [-1, 1]


## 12. Lỗi người mới hay gặp

### Lỗi 1: Bắt đầu cặp từ 0

```python
for x in range(n // 2):       # x = 0, 1, ...
    result += [x, -x]         # cặp (0, 0) trùng!
```

**Sửa:** `range(1, n // 2 + 1)`.

In [24]:
def wrong_start_zero(n):
    result = []
    for x in range(n // 2):
        result += [x, -x]
    if n % 2:
        result.append(0)
    return result

print(wrong_start_zero(4), is_valid(wrong_start_zero(4), 4))
print(sum_zero_pairs(4), is_valid(sum_zero_pairs(4), 4))

[0, 0, 1, -1] False
[1, -1, 2, -2] True


### Lỗi 2: Luôn thêm số 0 (kể cả khi `n` chẵn)

Khi `n` chẵn, các cặp đã đủ `n` số; thêm `0` làm mảng dài `n + 1`.

### Lỗi 3: Test bằng `==` với output mẫu

`assert Solution().sumZero(5) == [-7,-1,1,3,4]` sẽ **fail** dù lời giải đúng. Bài "return any" phải test bằng validator.

In [25]:
def wrong_always_zero(n):
    result = []
    for x in range(1, n // 2 + 1):
        result += [x, -x]
    result.append(0)          # quên kiểm tra n lẻ
    return result

print("n=4:", wrong_always_zero(4), "len =", len(wrong_always_zero(4)), is_valid(wrong_always_zero(4), 4))
print("Output của ta cho n=5:", Solution().sumZero(5), "== mẫu đề?", Solution().sumZero(5) == [-7, -1, 1, 3, 4])
print("...nhưng vẫn hợp lệ:", is_valid(Solution().sumZero(5), 5))

n=4: [1, -1, 2, -2, 0] len = 5 False
Output của ta cho n=5: [1, -1, 2, -2, 0] == mẫu đề? False
...nhưng vẫn hợp lệ: True


### Lỗi 4: Dùng `range(-k, k)` cho `n` lẻ

`range(-2, 2)` → `-2, -1, 0, 1`: thiếu `2`, tổng `-2`. Nhớ `stop = k + 1`.

In [26]:
print(list(range(-2, 2)), sum(range(-2, 2)))
print(list(range(-2, 3)), sum(range(-2, 3)))

[-2, -1, 0, 1] -2
[-2, -1, 0, 1, 2] 0


## 13. Mẹo nhận dạng pattern

- Đề có cụm **"return any ..."**, **"any valid answer"** ⇒ bài *constructive*: hãy tìm **cấu trúc đơn giản nhất** thoả điều kiện, đừng đi tìm kiếm.
- Điều kiện "tổng bằng 0" ⇒ nghĩ ngay tới **đối xứng** `x` / `-x`.
- Điều kiện "phân biệt / unique" ⇒ dùng dãy tăng nghiêm ngặt hoặc kiểm tra bằng `set`.
- Khi test, luôn viết **validator** thay vì so sánh với output mẫu.
- Tách bài theo **chẵn / lẻ** (`n // 2`, `n % 2`) là mẹo rất hay gặp.

## 14. Biến thể

1. **Tổng bằng `target`** thay vì 0: lấy mảng tổng 0 rồi cộng `target` vào **một** phần tử (ví dụ phần tử lớn nhất, để giữ unique).
2. **Chỉ dùng số dương, tổng bằng `S`**: `1, 2, ..., n-1` rồi số cuối `S - tổng`, cần `S - tổng > n - 1`.
3. **Mảng phải được sắp xếp tăng**: cách 4 (`range(1-n, n, 2)`) đã tự sắp xếp.

In [27]:
def sum_target(n: int, target: int) -> list[int]:
    arr = list(range(1 - n, n, 2))   # tăng dần, tổng 0
    arr[-1] += target                # tăng phần tử lớn nhất -> vẫn lớn nhất, vẫn unique (nếu target >= 0)
    return arr

for n, t in [(3, 10), (4, 7), (1, 5)]:
    arr = sum_target(n, t)
    print(n, t, arr, "sum =", sum(arr), "unique?", len(set(arr)) == n)

3 10 [-2, 0, 12] sum = 10 unique? True
4 7 [-3, -1, 1, 10] sum = 7 unique? True
1 5 [5] sum = 5 unique? True


## 15. Bài tập tự luyện (TODO — tăng dần)

**Bài 1 (tính tay):** Với `n = 6`, lời giải cặp đối xứng trả về gì? Còn `range(1 - n, n, 2)` trả về gì?

**Bài 2 (điền chỗ trống):** hoàn thiện `exercise_sum_zero` theo cách cặp đối xứng.

**Bài 3 (tự viết validator):** viết `exercise_is_valid(arr, n)`.

**Bài 4 (biến thể):** mảng `n` số unique có tổng `target`, xử lý cả `target` âm.

**Bài 5 (mini challenge):** mảng `n` số **dương** đôi một khác nhau có tổng **nhỏ nhất có thể**; trả về tổng nhỏ nhất đó và mảng.

**Bài 6 (nâng cao):** mảng `n` số unique tổng 0, thêm yêu cầu **trị tuyệt đối lớn nhất nhỏ nhất có thể**. Trị tuyệt đối lớn nhất tối thiểu là bao nhiêu?

In [28]:
def exercise_sum_zero(n: int) -> list[int]:
    result = []
    # TODO 1: duyệt x từ 1 đến n // 2
    # TODO 2: thêm x và -x
    # TODO 3: nếu n lẻ, thêm 0
    pass

# assert is_valid(exercise_sum_zero(5), 5)

In [29]:
def exercise_is_valid(arr: list[int], n: int) -> bool:
    # TODO: kiểm tra đủ 3 điều kiện của đề
    pass

# assert exercise_is_valid([-1, 0, 1], 3) is True
# assert exercise_is_valid([0, 0], 2) is False

In [30]:
def exercise_sum_target(n: int, target: int) -> list[int]:
    # TODO: n số khác nhau, tổng = target (target có thể âm)
    pass

# for n in range(1, 50):
#     for t in range(-20, 21):
#         arr = exercise_sum_target(n, t)
#         assert len(arr) == n and len(set(arr)) == n and sum(arr) == t

In [31]:
def exercise_min_positive(n: int) -> tuple[int, list[int]]:
    # TODO: trả về (tổng nhỏ nhất, mảng n số dương khác nhau đạt tổng đó)
    pass

def exercise_min_max_abs(n: int) -> int:
    # TODO: trị tuyệt đối lớn nhất nhỏ nhất có thể của mảng n số unique tổng 0
    pass

# assert exercise_min_positive(4) == (10, [1, 2, 3, 4])
# assert [exercise_min_max_abs(n) for n in range(1, 7)] == [0, 1, 1, 2, 2, 3]

## 16. Gợi ý 3 mức

**Bài 1** — (1) `n // 2 = 3` cặp. (2) cặp là `±1, ±2, ±3`. (3) bước 2 bắt đầu từ `1 - 6 = -5`.

**Bài 3** — (1) 3 điều kiện. (2) `len`, `set`, `sum`. (3) `len(arr) == n and len(set(arr)) == n and sum(arr) == 0`.

**Bài 4** — (1) Tạo mảng tổng 0 trước. (2) Cộng `target` vào phần tử lớn nhất nếu `target >= 0`, vào phần tử nhỏ nhất nếu `target < 0`. (3) Phần tử lớn nhất cộng thêm số không âm vẫn lớn nhất → không trùng.

**Bài 5** — (1) Số dương nhỏ nhất khác nhau là gì? (2) `1, 2, ..., n`. (3) Tổng `n(n+1)/2`.

**Bài 6** — (1) Có `n` số nguyên khác nhau trong đoạn `[-M, M]` thì cần `2M + 1 >= n`. (2) Với `n` chẵn không dùng được 0 một cách cân bằng? Thử `n = 2, 4`. (3) Đáp án là `n // 2` — thử đối chiếu với cách 2.

## 17. Đáp án tham khảo (tự làm trước rồi mới xem)

**Bài 1:** cặp đối xứng → `[1, -1, 2, -2, 3, -3]`; `range(-5, 6, 2)` → `[-5, -3, -1, 1, 3, 5]`.

In [32]:
def answer_sum_zero(n):
    result = []
    for x in range(1, n // 2 + 1):
        result += [x, -x]
    if n % 2:
        result.append(0)
    return result

def answer_is_valid(arr, n):
    return len(arr) == n and len(set(arr)) == n and sum(arr) == 0

def answer_sum_target(n, target):
    arr = list(range(1 - n, n, 2))
    if target >= 0:
        arr[-1] += target     # phần tử lớn nhất càng lớn hơn
    else:
        arr[0] += target      # phần tử nhỏ nhất càng nhỏ hơn
    return arr

def answer_min_positive(n):
    arr = list(range(1, n + 1))
    return n * (n + 1) // 2, arr

def answer_min_max_abs(n):
    # cách cặp đối xứng dùng trị tuyệt đối lớn nhất = n // 2;
    # không thể nhỏ hơn vì [-M, M] chỉ có 2M+1 số nguyên, nếu M < n//2 thì 2M+1 < n (n chẵn: 2M+1 <= n-1)
    return n // 2

print(answer_sum_zero(6), list(range(-5, 6, 2)))
for n in range(1, 200):
    assert answer_is_valid(answer_sum_zero(n), n)
    assert answer_min_positive(n)[0] == sum(answer_min_positive(n)[1])
    assert max(map(abs, answer_sum_zero(n))) == answer_min_max_abs(n)
    for t in range(-20, 21):
        arr = answer_sum_target(n, t)
        assert len(arr) == n and len(set(arr)) == n and sum(arr) == t
assert answer_is_valid([-1, 0, 1], 3) and not answer_is_valid([0, 0], 2)
assert answer_min_positive(4) == (10, [1, 2, 3, 4])
assert [answer_min_max_abs(n) for n in range(1, 7)] == [0, 1, 1, 2, 2, 3]
print("Tất cả đáp án PASS!")

[1, -1, 2, -2, 3, -3] [-5, -3, -1, 1, 3, 5]


Tất cả đáp án PASS!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def sumZero(self, n: int) -> list[int]:
        result = []
        for x in range(1, n // 2 + 1):   # bắt đầu từ 1, không phải 0
            result.append(x)
            result.append(-x)
        if n % 2 == 1:                    # lẻ thì thêm 0
            result.append(0)
        return result
```

Hoặc một dòng: `return list(range(1 - n, n, 2))`.

**Checklist trước khi submit:**
- [ ] Đúng tên method `sumZero`, trả về `list`.
- [ ] Cặp bắt đầu từ `1` (tránh `(0, 0)`).
- [ ] Chỉ thêm `0` khi `n` lẻ.
- [ ] Đã test `n = 1`, `n = 2`, `n = 1000` bằng **validator**, không so `==` với output mẫu.